# Lab 2 - Practical Tasks & OOP Framework

## Helper Functions & Error Metric

In [2]:
import random

def calc_01_error(y_true, y_pred):
    errors = sum(1 for yt, yp in zip(y_true, y_pred) if yt != yp)
    return errors / len(y_true)

## OOP Classifier Framework

In [3]:
class BaseClassifier:
    def fit(self, X, y):
        pass

    def predict(self, X):
        pass

    def score(self, X, y):
        preds = self.predict(X)
        return sum(1 for actual, pred in zip(y, preds) if actual == pred) / len(y)

    def error(self, X, y):
        return 1.0 - self.score(X, y)

## Classifier Implementations

In [4]:
class MajorityVoteClassifier(BaseClassifier):
    def __init__(self):
        self.dominant_class = None

    def fit(self, X, y):
        label_counts = {}
        for label in y:
            label_counts[label] = label_counts.get(label, 0) + 1
        self.dominant_class = max(label_counts, key=label_counts.get)
        return self

    def predict(self, X):
        return [self.dominant_class] * len(X)


class MemorizerClassifier(BaseClassifier):
    def __init__(self):
        self.lookup_table = {}
        self.unique_labels = []

    def fit(self, X, y):
        self.lookup_table.clear()
        self.unique_labels = list(set(y))
        for row, label in zip(X, y):
            self.lookup_table[tuple(row)] = label
        return self

    def predict(self, X):
        results = []
        for row in X:
            key = tuple(row)
            if key in self.lookup_table:
                results.append(self.lookup_table[key])
            else:
                results.append(random.choice(self.unique_labels))
        return results


class DecisionStumpClassifier(BaseClassifier):
    def __init__(self, feature_idx=0):
        self.feature_idx = feature_idx
        self.branch_rules = {}
        self.fallback = None

    def fit(self, X, y):
        grouped = {}
        for row, target in zip(X, y):
            val = row[self.feature_idx]
            grouped.setdefault(val, []).append(target)

        self.branch_rules = {
            val: max(set(targets), key=targets.count)
            for val, targets in grouped.items()
        }
        self.fallback = max(set(y), key=y.count)
        return self

    def predict(self, X):
        return [
            self.branch_rules.get(row[self.feature_idx], self.fallback)
            for row in X
        ]

## Task 1: Prepare the Dataset

In [5]:
X_train = [
    ["Y", "N", "N", "N"],
    ["N", "Y", "N", "N"],
    ["Y", "Y", "N", "N"],
    ["Y", "N", "Y", "Y"],
    ["N", "Y", "Y", "N"]
]
y_train = ["-", "-", "+", "-", "+"]

X_test = [
    ["Y", "Y", "Y", "N"],
    ["N", "N", "N", "N"],
    ["Y", "N", "N", "Y"]
]
y_test = ["+", "-", "-"]

print(f"Loaded {len(X_train)} training rows and {len(X_test)} test rows.")

Loaded 5 training rows and 3 test rows.


## Task 2: Implement Majority Vote

In [6]:
mv_clf = MajorityVoteClassifier()
mv_clf.fit(X_train, y_train)

train_preds = mv_clf.predict(X_train)
test_preds = mv_clf.predict(X_test)

print("Dominant class:", mv_clf.dominant_class)
print("Training error:", calc_01_error(y_train, train_preds))
print("Test error:", calc_01_error(y_test, test_preds))

Dominant class: -
Training error: 0.4
Test error: 0.3333333333333333


## Task 3: Implement Memorizer

In [7]:
mem_clf = MemorizerClassifier()
mem_clf.fit(X_train, y_train)

train_preds_mem = mem_clf.predict(X_train)
print("Stored examples:", mem_clf.lookup_table)
print("Training error (expected 0.0):", calc_01_error(y_train, train_preds_mem))

unseen_samples = [
    ["N", "N", "Y", "Y"],
    ["Y", "Y", "N", "Y"]
]
print("Predictions on unseen instances:", mem_clf.predict(unseen_samples))

Stored examples: {('Y', 'N', 'N', 'N'): '-', ('N', 'Y', 'N', 'N'): '-', ('Y', 'Y', 'N', 'N'): '+', ('Y', 'N', 'Y', 'Y'): '-', ('N', 'Y', 'Y', 'N'): '+'}
Training error (expected 0.0): 0.0
Predictions on unseen instances: ['+', '+']


## Task 4: Implement Decision Stump

In [8]:
total_features = len(X_train[0])
for f_idx in range(total_features):
    stump = DecisionStumpClassifier(feature_idx=f_idx)
    stump.fit(X_train, y_train)
    err = calc_01_error(y_train, stump.predict(X_train))
    print(f"Feature {f_idx} -> Rules: {stump.branch_rules} | Training Error: {err:.2f}")

Feature 0 -> Rules: {'Y': '-', 'N': '+'} | Training Error: 0.40
Feature 1 -> Rules: {'N': '-', 'Y': '+'} | Training Error: 0.20
Feature 2 -> Rules: {'N': '-', 'Y': '+'} | Training Error: 0.40
Feature 3 -> Rules: {'N': '+', 'Y': '-'} | Training Error: 0.40


## Task 5: Compare the Algorithms

In [9]:
benchmarks = {
    "Majority Vote": MajorityVoteClassifier().fit(X_train, y_train),
    "Memorizer": MemorizerClassifier().fit(X_train, y_train),
    "Decision Stump (Feature 2)": DecisionStumpClassifier(feature_idx=2).fit(X_train, y_train)
}

for model_name, model_obj in benchmarks.items():
    tr_err = calc_01_error(y_train, model_obj.predict(X_train))
    te_err = calc_01_error(y_test, model_obj.predict(X_test))
    print(f"{model_name}")
    print(f"  Training Error: {tr_err:.2f}")
    print(f"  Test Error:     {te_err:.2f}\n")

Majority Vote
  Training Error: 0.40
  Test Error:     0.33

Memorizer
  Training Error: 0.00
  Test Error:     0.67

Decision Stump (Feature 2)
  Training Error: 0.40
  Test Error:     0.00



## Class Task: Evaluation on Larger Dataset

In [10]:
random.seed(123)
sample_count = 250
feature_count = 5

dataset_X = []
dataset_y = []

for _ in range(sample_count):
    features = [random.choice(["Y", "N"]) for _ in range(feature_count)]
    num_y = features.count("Y")
    target = "+" if num_y >= 3 else "-"
    # add some noise
    if random.random() < 0.12:
        target = "-" if target == "+" else "+"
    dataset_X.append(features)
    dataset_y.append(target)

cutoff = int(sample_count * 0.75)
X_train_big, X_test_big = dataset_X[:cutoff], dataset_X[cutoff:]
y_train_big, y_test_big = dataset_y[:cutoff], dataset_y[cutoff:]

# Search best decision stump on training set
optimal_stump = None
min_error = float("inf")

for f_idx in range(feature_count):
    candidate_stump = DecisionStumpClassifier(feature_idx=f_idx)
    candidate_stump.fit(X_train_big, y_train_big)
    current_err = candidate_stump.error(X_train_big, y_train_big)
    if current_err < min_error:
        min_error = current_err
        optimal_stump = candidate_stump

eval_models = {
    "Majority Vote": MajorityVoteClassifier().fit(X_train_big, y_train_big),
    "Memorizer": MemorizerClassifier().fit(X_train_big, y_train_big),
    "Best Decision Stump": optimal_stump
}

print(f"Best decision stump selected feature index: {optimal_stump.feature_idx}\n")
for name, clf in eval_models.items():
    print(f"{name}:")
    print(f"  Train Error: {clf.error(X_train_big, y_train_big):.4f}")
    print(f"  Test Error:  {clf.error(X_test_big, y_test_big):.4f}\n")

Best decision stump selected feature index: 2

Majority Vote:
  Train Error: 0.4866
  Test Error:  0.4603

Memorizer:
  Train Error: 0.1711
  Test Error:  0.1587

Best Decision Stump:
  Train Error: 0.3102
  Test Error:  0.3968

